In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nifty100.db")

profit_loss = pd.read_sql("SELECT * FROM profitandloss", conn)

balance_sheet = pd.read_sql("SELECT * FROM balancesheet", conn)

financial_ratios = pd.read_sql("SELECT * FROM financial_ratios", conn)

print(profit_loss.shape)
print(balance_sheet.shape)
print(financial_ratios.shape)

(1164, 15)
(1140, 13)
(1164, 20)


In [5]:
companies = ["ABB", "ADANIPORTS", "ASIANPAINT"]

pl = profit_loss[
    (profit_loss["company_id"].isin(companies))
    & (profit_loss["year"] == "2024-03")
]

bs = balance_sheet[
    (balance_sheet["company_id"].isin(companies))
    & (balance_sheet["year"] == "2024-03")
]

data = pd.merge(pl, bs, on=["company_id", "year"])

data["manual_roe"] = data["net_profit"] / (data["equity_capital"] + data["reserves"]) * 100

ratio = financial_ratios[
    (financial_ratios["company_id"].isin(companies))
    & (financial_ratios["year"] == "2024-03")
]

result = pd.merge(
    data,
    ratio[["company_id", "return_on_equity_pct"]],
    on="company_id",
)

print(result[["company_id", "manual_roe", "return_on_equity_pct"]])

   company_id  manual_roe  return_on_equity_pct
0         ABB   32.468235             32.468235
1  ADANIPORTS   15.354883             15.354883
2  ASIANPAINT   29.677488             29.677488


In [6]:
companies = ["ABB", "ADANIPORTS", "ASIANPAINT"]

sales = profit_loss[
    profit_loss["company_id"].isin(companies)
]

sales = sales.sort_values(["company_id", "year"])

for company in companies:

    temp = sales[sales["company_id"] == company]

    start = temp[temp["year"] == "2019-03"]["sales"].values[0]
    end = temp[temp["year"] == "2024-03"]["sales"].values[0]

    cagr = ((end / start) ** (1 / 5) - 1) * 100

    engine = financial_ratios[
        (financial_ratios["company_id"] == company)
        & (financial_ratios["year"] == "2024-03")
    ]["revenue_cagr_5yr"].values[0]

    print(company)
    print("Manual CAGR :", round(cagr, 2))
    print("Engine CAGR :", engine)
    print()

ABB
Manual CAGR : 9.72
Engine CAGR : 9.72

ADANIPORTS
Manual CAGR : 19.58
Engine CAGR : 19.58

ASIANPAINT
Manual CAGR : 13.03
Engine CAGR : 13.03



### Sprint 2 Validation Summary

1. Validated ROE for ABB, ADANIPORTS and ASIANPAINT.
2. Validated 5-Year Revenue CAGR for the same companies.
3. Manual calculations matched database values exactly.
4. Difference observed: 0%.
5. financial_ratios table contains 1164 records.
6. ratio_edge_cases.log generated successfully.
7. Sprint 2 QA completed successfully.